# Earth Departure

Goals:
- Departure Geometry (rla, dla)
- Escape orbit and burn
- Parking orbit and location
- Launch site and opportunities
- Propagation

In [12]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [107]:
from tqdm import tqdm
from cesium.constants import DAY_TO_SEC
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from cesium.constants import DEG_TO_RAD, M_TO_KM, RAD_TO_DEG, KM_TO_M, HOUR_TO_SEC
from cesium.math.rotations import wrap_to_2pi
from cesium.math.time import jd_to_greg
from cesium.math.base import unit
from cesium.orbital.interplanetary import departure_characteristics
from cesium.guidance.interplanetary import interplanetary_circ_injection
from cesium.physics.dynamics import two_body
from cesium.orbital.transformations import ECI_from_HCI, itrf_to_gcrs_matrix, geodetic_LLA_to_ECEF, gcrs_to_itrf
from cesium.plotting import plot_3d
from cesium.world.constants import R_EARTH, MU_EARTH
from numpy.lib.npyio import NpzFile
from numpy.linalg import norm


## Loading trajectory
From `interplanetary.ipynb`

In [114]:

MISSION_NAME = "birthday-christmas"
mission: NpzFile = np.load(f"trajectories/{MISSION_NAME}.mission", allow_pickle=False)

def print_mission(mission: NpzFile):
    """Print dates, duration, sampling, units, and stored fields from a mission archive.

    Parameters
    ----------
    mission : NpzFile
        Loaded Mars Courier .mission archive in NumPy NPZ format. Expected
        fields include lambert_departure_jd, lambert_arrival_jd, tof_s,
        time_s, dt_s, frame, position_units, velocity_units, and time_origin.
        Epochs are heliocentric Lambert references, not liftoff or SOI crossings.

    Notes
    -----
    Uses the notebook's MISSION_NAME as the display title. State-array shapes
    and dtypes are listed without changing or closing the loaded archive.
    """
    print(f"Mission: {MISSION_NAME}")
    print("Heliocentric reference times")
    for label, key in (("Lambert departure (Earth reference)", "lambert_departure_jd"), ("Lambert arrival (Mars reference)", "lambert_arrival_jd")):
        print(f"{label}: " + "{:04.0f}-{:02.0f}-{:02.0f} {:02.0f}:{:02.0f}:{:06.3f}".format(*jd_to_greg(float(mission[key]))))
    print(f"Flight duration: {float(mission['tof_s']) / 86400:.4f} days")
    print(f"Samples: {mission['time_s'].size} (seconds since {mission['time_origin'].item()})")
    # print(f"Timesteps ranging from {mission['dt_s'].min():.3f}-{mission['dt_s'].max():.3f} s ")
    print(f"Frame: {mission['frame'].item()}; positions: {mission['position_units'].item()}; velocities: {mission['velocity_units'].item()}")
    print("\nStored fields:")
    for key in mission.files:
        info = mission[key].dtype

        if mission[key].dtype != float:
            info = '"' + mission[key] + '"'
        elif mission[key].shape == ():
            info = mission[key]
            
        # info = mission[key].dtype if mission[key].dtype == float else mission[key]
        print(f"  {key:24s} {mission[key].shape!s:14s} {info}")
print_mission(mission)


rv_sat_eci_km = (mission["satellite_rv_hci_km"] - mission["earth_rv_hci_km"]) @ np.kron(np.eye(2), ECI_from_HCI()).T
r_lambert_start_eci_km = rv_sat_eci_km[0,0:3]
v_inf_eci_km = rv_sat_eci_km[0,3:6]
lambert_dep_jd = float(mission["lambert_departure_jd"])

c3_km2, rla, dla = departure_characteristics(v_inf_eci_km)
print(f"\nEarth departure target: v_inf = {norm(v_inf_eci_km):.4f} km/s; C3 = {c3_km2:.4f} km²/s²")
print(f"RLA = {rla * RAD_TO_DEG:.4f} deg; DLA = {dla * RAD_TO_DEG:.4f} deg")


Mission: birthday-christmas
Heliocentric reference times
Lambert departure (Earth reference): 2035-06-13 23:25:00.000
Lambert arrival (Mars reference): 2035-12-25 00:00:00.000
Flight duration: 194.0243 days
Samples: 1942 (seconds since lambert_departure)
Frame: HCI; positions: km; velocities: km/s

Stored fields:
  lambert_departure_jd     ()             2464492.4756944445
  lambert_arrival_jd       ()             2464686.5
  tof_s                    ()             16763699.99999553
  time_s                   (1942,)        float64
  dt_s                     (1941,)        float64
  earth_rv_hci_km          (1942, 6)      float64
  mars_rv_hci_km           (1942, 6)      float64
  satellite_rv_hci_km      (1942, 6)      float64
  frame                    ()             "HCI"
  position_units           ()             "km"
  velocity_units           ()             "km/s"
  time_origin              ()             "lambert_departure"
  trajectory_model         ()             "heliocentric_

In [32]:
# ECEF stuff. Launch site position at Lambert departure
# TODO: iterate until I get the best dV
dcm_ecef_wrt_eci0 = itrf_to_gcrs_matrix(lambert_dep_jd).T

# Cape Canaveral
LAUNCH_LAT_DEG = 28.48889
LAUNCH_LON_DEG = -80.57778
r_cape_ecef_m = geodetic_LLA_to_ECEF(LAUNCH_LAT_DEG, LAUNCH_LON_DEG)

# Orbit
PARKING_ORBIT_ALT_M = 300e3  # m

The plot below has the Sun vector for reference. Here's where you can really see that it's 11:25p in June.

## Departure Characteristics

In [16]:
# ECI
C3_km2, rla, dla = departure_characteristics(v_inf_eci_km)


periapse = R_EARTH + PARKING_ORBIT_ALT_M
dv_p, ta_asymptote, aiming_radius, e = interplanetary_circ_injection(norm(v_inf_eci_km * KM_TO_M), periapse, MU_EARTH, deg=False)

print("Departure target (Lambert reference epoch)")
print(f"  v_inf: {norm(v_inf_eci_km):.4f} km/s; C3: {C3_km2:.4f} km²/s²")
print(f"  RLA: {rla * RAD_TO_DEG:.4f} deg; DLA: {dla * RAD_TO_DEG:.4f} deg")
print("\nCircular parking orbit / ideal tangential escape burn")
print(f"  Parking altitude: {PARKING_ORBIT_ALT_M * M_TO_KM:.1f} km")
print(f"  Periapsis radius: {periapse * M_TO_KM:.3f} km from Earth's center")
print(f"  Circular speed: {np.sqrt(MU_EARTH / periapse) * M_TO_KM:.4f} km/s")
print(f"  Post-burn periapsis speed: {np.sqrt(norm(v_inf_eci_km * KM_TO_M)**2 + 2 * MU_EARTH / periapse) * M_TO_KM:.4f} km/s")
print(f"  Required burn delta-v: {dv_p * M_TO_KM:.4f} km/s")
print("\nEscape hyperbola")
print(f"  Eccentricity: {e:.6f}")
print(f"  Asymptotic true anomaly: {ta_asymptote*RAD_TO_DEG:.4f} deg")
print(f"  Aiming radius (impact parameter): {aiming_radius * M_TO_KM:.3f} km")

Departure target (Lambert reference epoch)
  v_inf: 3.2647 km/s; C3: 10.6580 km²/s²
  RLA: 4.9947 deg; DLA: 4.1715 deg

Circular parking orbit / ideal tangential escape burn
  Parking altitude: 300.0 km
  Periapsis radius: 6678.138 km from Earth's center
  Circular speed: 7.7258 km/s
  Post-burn periapsis speed: 11.4032 km/s
  Required burn delta-v: 3.6774 km/s

Escape hyperbola
  Eccentricity: 1.178564
  Asymptotic true anomaly: 148.0479 deg
  Aiming radius (impact parameter): 23326.193 km


In [ ]:
# Making it 3D

# orbit vectors in eci
v_vec = unit(v_inf_eci_km)
n_vec = unit(np.cross(v_inf_eci_km, dcm_ecef_wrt_eci0.T @ r_cape_ecef_m))
right_vec = unit(np.cross(n_vec, v_inf_eci_km))

# I like dealing in column vectors
dcm_eci_wrt_orbit = np.stack((v_vec, right_vec, n_vec), axis=1)
assert np.linalg.det(dcm_eci_wrt_orbit) == 1.0 # actually works

r_p_orbit = np.array([np.cos(ta_asymptote), -np.sin(ta_asymptote), 0]) * periapse
r_p_eci = dcm_eci_wrt_orbit @ r_p_orbit
v_p_eci = unit(np.cross(n_vec, r_p_eci)) * (
    np.sqrt(MU_EARTH/norm(r_p_orbit)) + dv_p
)

## Propagating escape

In [36]:
state0 = np.concatenate((r_p_eci, v_p_eci))

t_traj_s = np.linspace(0, 12*HOUR_TO_SEC, 10000)
departure_eci = solve_ivp(
    lambda _, s: two_body(s, MU_EARTH),
    (t_traj_s[0], t_traj_s[-1]),
    state0,
    t_eval=t_traj_s,
    rtol=1e-9,
    atol=1e-12,
)
assert departure_eci.success

r_traj_eci_m = departure_eci.y[0:3].T

Perpendicular distance to v_inf comparison (km)
  h/v_inf: 23326.193351 (0.000019 km s.d. very small)
  Theoretical aiming radius [km]: 23326.193346
  diff [m]: 0.005624058441817761 (0.018813752049209678 s.d. very small)


## Visualization

In [104]:
# Translate by Earth's rv at each sample, then rotate HCI into ECI.
sun_direction_eci = ECI_from_HCI() @ (-mission["earth_rv_hci_km"][0, :3])

frames = [
    {"name": "ECI", "location": [0, 0, 0],
            "dcm_body_wrt_hci": np.eye(3), "length": 10000},
    # {"name": "ECEF", "location": [0, 0, 0],
    #         "dcm_body_wrt_hci": dcm_ecef_wrt_eci, "length": 10000},  # just for visualization
]

vectors = [
    {"direction": sun_direction_eci, "length": 20000,
             "name": "Sun direction", "color": "goldenrod", "label": True},
    {"direction": r_p_eci, "length": 10000,
                 "name": "periapse", "color": "orange", "label": True},

    {"direction": n_vec, "length": 10000,
                 "name": "Normal vec", "color": "black", "label": True},  # just for visualization
    {"direction": right_vec, "length": 10000,
                 "name": "right_vec", "color": "black", "label": True},  # just for visualization
]

view_range = 250000
fig_earth = plot_3d(
    [rv_sat_eci_km[:,0:3], r_traj_eci_m * M_TO_KM], [mission["time_s"], t_traj_s],
    labels=["Lambert Departure Direction","Trajectory"], trace_colors=["purple", "black"],
    bodies={"name": "Earth", "radius": R_EARTH*M_TO_KM, "color": "royalblue"},
    frames=frames,
    vectors=vectors,
    title=f"Earth departure — {view_range} km view", units="km",
    azim=80, elev=30, show_end=False, show=False,
)
fig_earth.update_scenes(
    xaxis_range=[-view_range, view_range], # km
    yaxis_range=[-view_range, view_range], # km
    zaxis_range=[-view_range, view_range], # km
)
r_cape_eci_km = (dcm_ecef_wrt_eci0.T @ r_cape_ecef_m) * M_TO_KM
fig_earth.add_scatter3d(
    x=[r_cape_eci_km[0], r_p_eci[0]*M_TO_KM],
    y=[r_cape_eci_km[1], r_p_eci[1]*M_TO_KM],
    z=[r_cape_eci_km[2], r_p_eci[2]*M_TO_KM],
    mode="markers+text", name="Cape Canaveral", text=["Cape Canaveral"],
    textposition="top center", marker={"size": 7, "color": "red"},
)
fig_earth.show()

## Validating

In [125]:
# Impact parameter / aiming distance

# h/v_inf
impact_parameter = norm(np.cross(departure_eci.y[:3].T, departure_eci.y[3:].T), axis=1) / norm(v_inf_eci_km * KM_TO_M)
print("Perpendicular distance to v_inf comparison (km)")
print(f"  h/v_inf: {impact_parameter.mean() * M_TO_KM:.6f} ({impact_parameter.std() * M_TO_KM:.6f} km s.d. very small)")
print(f"  Theoretical aiming radius [km]: {aiming_radius * M_TO_KM:.6f}")
diff = impact_parameter - aiming_radius
print(f"  diff [m]: {diff.mean()} ({diff.std()} s.d. very small)")

Perpendicular distance to v_inf comparison (km)
  h/v_inf: 23326.193351 (0.000019 km s.d. very small)
  Theoretical aiming radius [km]: 23326.193346
  diff [m]: 0.005624058441817761 (0.018813752049209678 s.d. very small)


In [126]:
# C3 at each point
c3_propagated = np.array(
    norm(departure_eci.y[3:].T, axis=1)**2
    - 2 * MU_EARTH / norm(r_traj_eci_m, axis=1)
) * M_TO_KM**2

diff_c3 = c3_propagated - c3_km2

print(f"Diff between `c3_propagated` and `c3_lambert` [km2]: {diff_c3.mean():.6e} ({diff_c3.std():.6e} s.d.)")

Diff between `c3_propagated` and `c3_lambert` [km2]: 3.716343e-08 (1.903529e-08 s.d.)


In [ ]:
# ECI -> ECEF (tqdm just in case)
t_traj_jd = lambert_dep_jd + t_traj_s / DAY_TO_SEC
rv_departure_ecef = np.empty_like(departure_eci.y.T)
for i in tqdm(range(len(t_traj_jd)), desc="ECI → ECEF"):
    state = departure_eci.y[:, i]
    rv_departure_ecef[i] = np.concatenate(gcrs_to_itrf(state[:3], state[3:6], t_traj_jd[i]))


ECI → ECEF: 100%|██████████| 10000/10000 [00:00<00:00, 53354.98it/s]


In [33]:
# Translate by Earth's rv at each sample, then rotate HCI into ECI.

frames = [
    {"name": "ECEF (launch)", "location": [0, 0, 0],
            "dcm_body_wrt_hci": np.eye(3), "length": 10000},
    # {"name": "ECEF", "location": [0, 0, 0],
    #         "dcm_body_wrt_hci": dcm_ecef_wrt_eci, "length": 10000},  # just for visualization
]

vectors = [
    {"direction": dcm_ecef_wrt_eci0@sun_direction_eci, "length": 20000,
             "name": "Sun direction", "color": "goldenrod", "label": True},
    {"direction": dcm_ecef_wrt_eci0@r_p_eci, "length": 10000,
                 "name": "periapse", "color": "orange", "label": True},

    # {"direction": n_vec, "length": 10000,
    #              "name": "Normal vec", "color": "black", "label": True},  # just for visualization
    # {"direction": right_vec, "length": 10000,
    #              "name": "right_vec", "color": "black", "label": True},  # just for visualization
]

view_range = 100000
fig_earth = plot_3d(
    [rv_departure_ecef[:,0:3] * M_TO_KM], [t_traj_s],
    labels=["Trajectory"], trace_colors=["black"],
    bodies={"name": "Earth", "radius": R_EARTH*M_TO_KM, "color": "royalblue"},
    frames=frames,
    vectors=vectors,
    title=f"Earth departure ECEF — {view_range} km view", units="km",
    azim=80, elev=30, show_end=False, show=False,
)
fig_earth.update_scenes(
    xaxis_range=[-view_range, view_range], # km
    yaxis_range=[-view_range, view_range], # km
    zaxis_range=[-view_range, view_range], # km
)
r_cape_eci_km = (r_cape_ecef_m) * M_TO_KM
fig_earth.add_scatter3d(
    x=[r_cape_eci_km[0], r_p_eci[0]*M_TO_KM],
    y=[r_cape_eci_km[1], r_p_eci[1]*M_TO_KM],
    z=[r_cape_eci_km[2], r_p_eci[2]*M_TO_KM],
    mode="markers+text", name="Cape Canaveral", text=["Cape Canaveral"],
    textposition="top center", marker={"size": 7, "color": "red"},
)
fig_earth.show()